# Sprint 1 · Análisis exploratorio de datos (EDA)

**Tesis:** Diagnóstico de causalidad de fallas en una red telco usando grafos · Región **Pasco**, Perú.

Objetivo del EDA (tarjeta Trello): **explorar la distribución de alarmas, las avalanchas y los patrones de
co-ocurrencia** relevantes para la causalidad. Reutiliza el pipeline de `src/ingesta.py` (no duplica código).

**Contenido**
1. Carga del consolidado
2. Estadística descriptiva por segmento / severidad
3. Análisis de avalancha (equipos/nodos con 3+ alarmas por ventana)
4. Matriz de co-ocurrencia de alarmas
5. Series temporales de llegada de alarmas
6. **Hallazgos accionables**


## 1. Carga del consolidado

Si ya ejecutaste el notebook 01 (o `python src/ingesta.py`), se reutiliza `data/processed/dataset_consolidado.parquet`.
Si no existe, se reconstruye al vuelo desde los datos crudos. Ajusta `BASE_DIR` solo si reconstruyes.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

BASE_DIR = str(ROOT / "data" / "raw")   # solo se usa si hay que reconstruir
proc = ROOT / "data" / "processed" / "dataset_consolidado.parquet"

if proc.exists():
    df = pd.read_parquet(proc)
    print("Cargado de:", proc)
else:
    from src.ingesta import construir_dataset, parsear_moi, marcar_particion_temporal
    df = marcar_particion_temporal(parsear_moi(construir_dataset(BASE_DIR)))
    print("Reconstruido desde:", BASE_DIR)

# garantiza columnas de preprocesado aunque se cargue un consolidado viejo
if "moi_puerto" not in df.columns:
    from src.ingesta import parsear_moi; df = parsear_moi(df)
if "split" not in df.columns:
    from src.ingesta import marcar_particion_temporal; df = marcar_particion_temporal(df)

# tipado defensivo de fechas
for col in ["firstOccurUtc", "arriveUtc", "occurUtc", "latestOccurUtc", "ventana_ts"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Dataset:", df.shape, "| ventanas:", df["ventana_id"].nunique())
df.head(3)

> **Resumen.** Carga el consolidado ya filtrado (solo equipos del inventario) y normaliza las fechas. Imprime el tamaño (filas × columnas) y el nº de ventanas: es la base sobre la que corre todo el EDA.

## 2. Estadística descriptiva por segmento / severidad

*(Subtarea: estadística descriptiva por tipo/severidad.)*

In [ ]:
print("Total de alarmas:", len(df))
print("Equipos (IPs) distintos:", df["address"].nunique() if "address" in df else "n/d")
print()
# por segmento de red
if "segmento_red" in df:
    print("Por segmento de red:"); print(df["segmento_red"].value_counts(dropna=False), "\n")
# por severidad
sev_col = "severidad_txt" if "severidad_txt" in df else ("severity" if "severity" in df else None)
if sev_col:
    print("Por severidad:"); print(df[sev_col].value_counts(dropna=False))

> **Resumen.** Conteos base del dataset: total de alarmas, nº de equipos distintos y cómo se reparten las alarmas por **segmento de red** (radio / router / transporte) y por **severidad**. Da la magnitud antes de graficar.

In [ ]:
# Figuras: alarmas por segmento y por severidad
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "segmento_red" in df:
    df["segmento_red"].value_counts().plot.bar(ax=axes[0], color="#4C78A8")
    axes[0].set_title("Alarmas por segmento de red"); axes[0].set_ylabel("n alarmas")
    axes[0].tick_params(axis="x", rotation=20)
if sev_col:
    orden = ["Crítica", "Mayor", "Menor", "Advertencia"]
    vc = df[sev_col].value_counts()
    vc = vc.reindex([o for o in orden if o in vc.index]) if sev_col == "severidad_txt" else vc
    vc.plot.bar(ax=axes[1], color="#E45756")
    axes[1].set_title("Alarmas por severidad"); axes[1].set_ylabel("n alarmas")
    axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

> **Qué muestra la gráfica.** Dos diagramas de barras: *(izquierda)* cuántas alarmas aporta cada **segmento de red** y *(derecha)* cuántas hay de cada **severidad**. Permite ver el **desbalance** entre segmentos y entre severidades, algo a tener en cuenta en la evaluación (un segmento o una severidad dominante sesga las métricas).

In [ ]:
# Cruce segmento x severidad y top de tipos de alarma
if "segmento_red" in df and sev_col:
    display(pd.crosstab(df["segmento_red"], df[sev_col], margins=True))
if "alarmName" in df:
    print("\nTop 15 tipos de alarma (alarmName):")
    display(df["alarmName"].value_counts().head(15).to_frame("n"))

> **Qué muestran las tablas.** La primera cruza **segmento × severidad** (cuántas alarmas de cada severidad hay en cada segmento). La segunda lista los **15 tipos de alarma** (`alarmName`) más frecuentes — el vocabulario dominante del dataset.

## 3. Análisis de avalancha

Una **avalancha** es la acumulación de muchas alarmas disparadas por un mismo evento. La medimos por
**equipo dentro de una ventana**: agrupamos `[ventana_id, address]` y contamos alarmas; un grupo con **3 o más**
se considera avalancha. Esto justifica el enfoque causal (hay que hallar el evento raíz que explica el resto).

*(Subtarea: análisis de avalancha — nodos con 3+ alarmas.)*

In [ ]:
KEY = "address"       # equipo; cambia a 'CODIGO DE NODO' si prefieres a nivel de nodo
UMBRAL = 3

g = (df.dropna(subset=[KEY])
       .groupby(["ventana_id", KEY]).size().rename("n_alarmas").reset_index())
aval = g[g["n_alarmas"] >= UMBRAL]
print(f"Grupos (equipo x ventana): {len(g):,}")
print(f"Grupos en avalancha (>= {UMBRAL}): {len(aval):,} ({len(aval)/max(len(g),1):.1%})")
print(f"Tamaño medio de avalancha: {aval['n_alarmas'].mean():.1f} | máx: {g['n_alarmas'].max()}")

fig, ax = plt.subplots(figsize=(8, 4))
g["n_alarmas"].plot.hist(bins=range(1, int(g["n_alarmas"].max()) + 2),
                         ax=ax, color="#72B7B2", edgecolor="white")
ax.axvline(UMBRAL - 0.5, color="#E45756", ls="--", label=f"umbral avalancha ({UMBRAL})")
ax.set_title("Distribución de alarmas por equipo y ventana")
ax.set_xlabel("n alarmas en el grupo"); ax.set_ylabel("n grupos"); ax.legend()
plt.tight_layout(); plt.show()

> **Qué muestra la gráfica.** Histograma del **nº de alarmas por grupo *equipo × ventana***. La línea roja marca el umbral de avalancha (3): todo lo que queda a su derecha son avalanchas. Confirma el problema de la tesis — el **29.7 %** de los grupos concentra 3+ alarmas, con picos de hasta **24** — un evento dispara una cascada.

In [ ]:
# Top 10 avalanchas (equipo+ventana con más alarmas)
print("Top 10 avalanchas:")
display(aval.sort_values("n_alarmas", ascending=False).head(10).reset_index(drop=True))

> **Qué muestra la tabla.** Los 10 grupos *equipo × ventana* con **más alarmas simultáneas** — los eventos de falla más severos del período, candidatos naturales a caso de estudio para la verdad de referencia.

## 4. Matriz de co-ocurrencia de alarmas

¿Qué tipos de alarma **aparecen juntos** en el mismo equipo y ventana? Una pareja frecuente puede ser una
**relación causa→efecto** (p. ej. señal baja → pérdida de trama) o simplemente **dos avisos del mismo evento**
(p. ej. dos alarmas que reportan la misma caída de una sesión BGP). La co-ocurrencia no distingue entre ambos casos;
eso lo hará el motor (Hawkes topológico) usando el orden temporal y la topología.

*(Subtarea: matriz de co-ocurrencia.)*

In [ ]:
from itertools import combinations
from collections import Counter

TOPN = 15
top_alarmas = df["alarmName"].value_counts().head(TOPN).index.tolist()
sub = df[df["alarmName"].isin(top_alarmas)]

pares = Counter()
for _, grp in sub.groupby(["ventana_id", "address"]):
    tipos = sorted(set(grp["alarmName"].dropna()))
    for a, b in combinations(tipos, 2):
        pares[(a, b)] += 1

idx = {a: i for i, a in enumerate(top_alarmas)}
M = np.zeros((len(top_alarmas), len(top_alarmas)), dtype=int)
for (a, b), n in pares.items():
    M[idx[a], idx[b]] += n; M[idx[b], idx[a]] += n

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(M, cmap="YlOrRd")
ax.set_xticks(range(len(top_alarmas))); ax.set_xticklabels(top_alarmas, rotation=90, fontsize=8)
ax.set_yticks(range(len(top_alarmas))); ax.set_yticklabels(top_alarmas, fontsize=8)
ax.set_title("Co-ocurrencia de tipos de alarma (mismo equipo y ventana)")
fig.colorbar(im, ax=ax, shrink=0.8, label="n de co-ocurrencias")
plt.tight_layout(); plt.show()

> **Qué muestra la gráfica.** Un **mapa de calor de co-ocurrencia**: cada celda cuenta cuántas veces dos tipos de alarma (ejes X/Y) aparecen juntos en el mismo equipo y ventana. Cuanto más intenso el color, más fuerte la asociación. Resalta el par **`BGP Status Changed` – `Bgp Peer Backward Transition`**: dos avisos que suelen acompañar la misma caída de sesión BGP en los routers.

In [ ]:
# Top 15 parejas que más co-ocurren
top_pares = (pd.DataFrame([(a, b, n) for (a, b), n in pares.items()],
                          columns=["alarma_A", "alarma_B", "co_ocurrencias"])
               .sort_values("co_ocurrencias", ascending=False).head(15).reset_index(drop=True))
display(top_pares)

> **Qué muestra la tabla.** Las 15 **parejas de alarmas que más co-ocurren**, ordenadas por frecuencia. La primera es `BGP Status Changed` + `Bgp Peer Backward Transition` (**1 946** veces): co-ocurren porque describen el mismo evento, no porque una cause a la otra. Ejemplo directo de que **correlación no es causalidad**.

## 5. Series temporales de llegada de alarmas

Cómo llegan las alarmas en el tiempo: los **picos** marcan eventos de falla (muchas alarmas en poco tiempo),
justo el fenómeno temporal que el proceso de Hawkes aprovecha para inferir causalidad.

*(Subtarea: series temporales de llegada.)*

In [ ]:
tcol = "firstOccurUtc" if ("firstOccurUtc" in df and df["firstOccurUtc"].notna().any()) else "ventana_ts"
serie = (df.dropna(subset=[tcol]).set_index(tcol).assign(n=1)["n"]
           .sort_index().resample("1h").sum())

fig, ax = plt.subplots(figsize=(12, 4))
serie.plot(ax=ax, color="#4C78A8")
ax.set_title(f"Llegada de alarmas por hora (campo: {tcol})")
ax.set_xlabel("tiempo"); ax.set_ylabel("n alarmas / hora")
plt.tight_layout(); plt.show()

print("Hora pico:", serie.idxmax(), "->", int(serie.max()), "alarmas")

> **Qué muestra la gráfica.** La **curva de llegada de alarmas por hora**. Los **picos** son eventos de falla (muchas alarmas en poco tiempo); los valles, operación normal. Debajo se imprime la **hora pico**. Esa dinámica de excitación temporal es la que explotará el proceso de Hawkes para ordenar causa y efecto.

In [ ]:
# Llegada por severidad (apiladas) si hay datos de tiempo y severidad
if sev_col and tcol in df:
    piv = (df.dropna(subset=[tcol]).assign(n=1)
             .groupby([pd.Grouper(key=tcol, freq="1h"), sev_col])["n"].sum()
             .unstack(fill_value=0))
    if len(piv):
        ax = piv.plot.area(figsize=(12, 4), alpha=0.8)
        ax.set_title("Llegada de alarmas por hora y severidad")
        ax.set_xlabel("tiempo"); ax.set_ylabel("n alarmas / hora")
        plt.tight_layout(); plt.show()

> **Qué muestra la gráfica.** La misma llegada por hora, pero **apilada por severidad** (áreas de color). Permite ver qué tipo de alarma domina en cada pico — por ejemplo, si un evento mayor está compuesto sobre todo por alarmas Críticas.

## 6. Hallazgos accionables

*(Subtarea: sección de hallazgos.)* La siguiente celda calcula un **resumen automático**; debajo se interpreta.

In [ ]:
resumen = {
    "ventanas": int(df["ventana_id"].nunique()),
    "alarmas_totales": int(len(df)),
    "equipos_distintos": int(df["address"].nunique()) if "address" in df else None,
    "pct_en_avalancha": round(len(aval) / max(len(g), 1), 3),
    "tam_avalancha_max": int(g["n_alarmas"].max()),
    "pct_con_puerto": round(df["moi_puerto"].notna().mean(), 3) if "moi_puerto" in df else None,
    "pct_sin_nodo": round(df["CODIGO DE NODO"].isna().mean(), 3) if "CODIGO DE NODO" in df else None,
    "pareja_top_coocurrencia": (top_pares.iloc[0]["alarma_A"], top_pares.iloc[0]["alarma_B"],
                                int(top_pares.iloc[0]["co_ocurrencias"])) if len(top_pares) else None,
    "hora_pico": str(serie.idxmax()),
}
import json; print(json.dumps(resumen, indent=2, ensure_ascii=False, default=str))

> **Resumen.** Diccionario con las **cifras clave del EDA** (ventanas, alarmas, % en avalancha, anclaje a puerto, pareja top de co-ocurrencia, hora pico). Es el insumo numérico que sustenta la interpretación de abajo.

### Interpretación

Sobre **313 ventanas** (unidad de análisis cada 15 min) y **51 533 alarmas** de **77 equipos de red**, tras aplicar
los dos filtros de alcance de la ingesta (solo equipos del inventario y sin sectoriales).

**H1 — La mayor parte de las alarmas era ruido de clientes.** Al separar los **sectoriales RTN510_AP** (radios
multipunto hacia clientes finales), el dataset pasó de 138 346 a 51 533 alarmas: **≈ 63 % del volumen**
venía de equipos que no afectan la conectividad del nodo *(cifra estimada por diferencia entre corridas; el valor
exacto lo informa el notebook 01)*. El operador del NOC ve ese ruido mezclado con las alarmas que sí importan, lo
que refuerza el problema de la tesis. Además, `% sin nodo = 0`: todas las alarmas restantes se ubican en un nodo.

**H2 — Hay avalanchas y son grandes.** El **29.7 %** de los grupos *equipo × ventana* concentra **3 o más**
alarmas (hasta **24** en un mismo equipo y ventana), lo que da **4 192 casos** a diagnosticar. Esto **confirma
empíricamente el problema**: un evento raíz dispara una avalancha → se necesita diagnóstico **causal**, no conteo.

**H3 — La co-ocurrencia dominante no es causal: son dos avisos del mismo evento.** La pareja más frecuente es
**`BGP Status Changed` + `Bgp Peer Backward Transition`** (**1 946** co-ocurrencias). Ambas reportan la caída de
una sesión BGP en los routers: aparecen juntas porque describen lo mismo, y la causa real suele estar *debajo*
(interfaz o enlace caído, falla de transporte o de energía). Dos consecuencias:
- **Corrección respecto a la versión anterior:** el patrón `RADIO_RSL_LOW` + `MW_LOF` que dominaba antes venía de
  los **sectoriales** (radios hacia clientes), no del backhaul. Al separarlos desaparece del dataset de causa raíz.
- Es evidencia directa de que **correlación ≠ causalidad**: el motor debe usar **topología + orden temporal**, y la
  ingeniería de atributos debe **agrupar alarmas sinónimas** (mismo evento con dos nombres) para no contarlas dos veces.

**H4 — Anclaje a puerto: 18.2 %.** Subió (antes 6.3 %) porque los sectoriales reportaban a nivel de equipo
(`NE=...`). Sigue siendo bajo y es coherente con la decisión de modelar el grafo a nivel de **equipo y enlace**,
con el puerto como **atributo dinámico de la alarma** (los puertos migran por averías).

**H5 — Hay picos temporales claros.** El máximo de llegada es el **2026-07-14 ≈ 06:00**. Esa dinámica de
excitación temporal es la que explota el proceso de Hawkes. → **Acción:** revisar ese pico con el NOC como caso
de estudio para la verdad de referencia.

**Implicación de diseño.** Las avalanchas confirman el problema y H3 muestra que la co-ocurrencia sola engaña:
se mantiene la decisión de un motor causal (**Hawkes topológico**, F4) sobre el grafo de la red (F3), y se agregan
dos tareas al Sprint 2: agrupar alarmas sinónimas y construir un **conjunto de referencia etiquetado** para medir.
